# 19 · Semantic Chunking

**Idea:** fixed-size chunking cuts on an arbitrary word count and routinely splits a single idea across two chunks. Semantic chunking places boundaries where the *meaning* shifts: embed each sentence, walk them in order, and start a new chunk when the next sentence drifts too far from the current chunk's running centroid. The result is topically coherent passages that embed and retrieve better.

> Embeds sentences with Ollama (`nomic-embed-text`). Set `provider=` to use a cloud embedder instead.

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
root = next(x for x in [p, *p.parents] if (x / 'rag_lab').exists())
sys.path.insert(0, str(root))

Load one document and compare a fixed/recursive split against a semantic split. Same text, different boundary logic.

In [ ]:
from rag_lab import NOTES_DIR, recursive_chunks, semantic_chunks

text = (NOTES_DIR / 'rag_overview.md').read_text(encoding='utf-8')

fixed = recursive_chunks(text, source='rag_overview.md', chunk_size=120, overlap=20)
semantic = semantic_chunks(text, source='rag_overview.md', threshold=0.5, provider='ollama')

print(f'recursive (fixed size) -> {len(fixed)} chunks')
print(f'semantic  (topic-based) -> {len(semantic)} chunks')

Each semantic chunk should read as one coherent thought. Inspect a few — notice they end at natural topic breaks, not mid-idea.

In [ ]:
for c in semantic[:3]:
    print(f'--- chunk {c.index} ({len(c.text)} chars) ---')
    print(c.text[:220], '...\n')

The `threshold` knob controls granularity: a **lower** threshold tolerates more drift, giving fewer/larger chunks; a **higher** threshold splits more eagerly.

In [ ]:
for t in (0.3, 0.5, 0.7):
    n = len(semantic_chunks(text, threshold=t, provider='ollama'))
    print(f'threshold={t} -> {n} chunks')

**Takeaway:** semantic chunking spends embedding calls up front to buy cleaner, self-contained passages — a good trade when retrieval precision matters more than ingestion speed. Tune `threshold` per corpus.

Next up → `20_query_transformation.ipynb`, where we rewrite the *query* (not the documents) to lift recall.